# What the counts refuse

The path keeps four predictions and one split.

The global mean on `ua.base` is $319153/90570$. On `ua.test` its RMSE is about 1.122. The user mean and the movie mean each reach about 1.04. The bias formula $\bar r_u + \bar r_i - \mu$ reaches about 0.992. Clipping to the interval from 1 to 5 changes 145 predictions and moves the error by about 0.0015.

The counts refuse a few tempting shortcuts.

An empty cell is not a zero. The grid has 1486126 empty cells, and the scale starts at 1.

A genre flag is not a rating. Drama is on 725 movies, and user 1's 5 stars for *Toy Story* is a separate fact.

A zip code is not a similarity. It is text on the user row.

`MSSubClass`-style mistakes have a cousin here: user id 196 is one writer, while 196 is also the number of students. The identifier and the tally are not the same column.

Movie 1582 and movie 1653 occur in `ua.test` and in no row of `ua.base`. A movie mean cannot be built for them from the training file. The bias formula falls back to the user mean. Dropping those two rows to make the table look complete would score a different test set.

The five folds `u1` through `u5` are a real protocol. This path does not average them into the 0.992. That 0.992 is one split: ten ratings per user, means fit on the rest.


In [2]:
# Locate the MovieLens tables beside this notebook, or under the course root.
from pathlib import Path

def data_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "16-Movie Recommendation" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "u.info").exists():
            return direct
    raise FileNotFoundError(name)

def read_lines(name):
    return data_path(name).read_text(encoding="latin-1").splitlines()

def read_ratings(name):
    rows = []
    for line in read_lines(name):
        if not line.strip():
            continue
        user, item, rating, stamp = line.split("\t")
        rows.append((int(user), int(item), int(rating), int(stamp)))
    return rows

ratings = read_ratings("u.data")
print(len(ratings), ratings[0])


100000 (196, 242, 3, 881250949)


In [3]:
# The fallback movies are real titles, and they are absent from ua.base.
movies = {int(row[0]): row[1] for row in (line.split("|") for line in read_lines("u.item"))}
base_items = {item for _, item, _, _ in read_ratings("ua.base")}
test_items = {item for _, item, _, _ in read_ratings("ua.test")}
missing = sorted(test_items - base_items)
print(missing, [movies[item] for item in missing])
assert missing == [1582, 1653]
assert movies[1582] == "T-Men (1947)"
assert movies[1653] == "Entertaining Angels: The Dorothy Day Story (1996)"
assert 1486126 == 943 * 1682 - 100000


[1582, 1653] ['T-Men (1947)', 'Entertaining Angels: The Dorothy Day Story (1996)']


## Pitfall

About 0.992 is a reproducible error on a named split of this file. It is not a leaderboard rank, and it is not the error of a neighbor model or a factorization. Those models answer a further question this path states and does not fit.

## Summary

Keep the bias formula, the `ua` split, and the empty cells as empty. The two movies with no base rating stay in the test score, predicted by the user mean alone.
